# Bài thực hành 1 — Phân tích Marketing Campaign bằng SweetViz

Notebook độc lập này đọc `marketing_campaign.csv`, kiểm tra chất lượng và cấu trúc dữ liệu, phân tích biến phản hồi chiến dịch (`Response`), sau đó tạo báo cáo HTML bằng SweetViz.

> **Lưu ý khi đọc dữ liệu:** tệp CSV này dùng dấu chấm phẩy (`;`) để phân cách cột. Ô đọc dữ liệu bên dưới đã đặt `sep=";"` để tránh gộp toàn bộ dữ liệu vào một cột.

Báo cáo SweetViz được lưu tại `reports/marketing_campaign_sweetviz.html`. Có thể mở tệp HTML bằng trình duyệt sau khi chạy notebook.

Nếu SweetViz chưa có trong môi trường Python, cài đặt trong một ô code bằng `%pip install sweetviz` rồi chạy lại notebook.


In [4]:
# Bước 1: nạp thư viện độc lập; notebook này không phụ thuộc biến từ notebook khác.
from pathlib import Path
import pandas as pd

# Bước 2: dò vị trí marketing_campaign.csv trong thư mục chạy hoặc thư mục Lab05.
data_directories = [
    Path.cwd(),
    Path.cwd() / "Lab05",
    Path(r"d:\ML\sgu26k2_ML\Lab05"),
]
BASE_DIR = next(
    (
        directory.resolve()
        for directory in data_directories
        if (directory / "marketing_campaign.csv").is_file()
    ),
    Path.cwd().resolve(),
)
MARKETING_CSV = BASE_DIR / "marketing_campaign.csv"

# Bước 3: báo lỗi ngay nếu file chưa ở vị trí được dò.
print(f"Thư mục dữ liệu: {BASE_DIR}")
if not MARKETING_CSV.is_file():
    raise FileNotFoundError(
        f"Không tìm thấy {MARKETING_CSV}. "
        "Đặt marketing_campaign.csv cạnh notebook hoặc cập nhật data_directories."
    )


Thư mục dữ liệu: D:\ML\sgu26k2_ML\Lab05


In [5]:
# Bước 1: đọc CSV phân cách bằng dấu chấm phẩy và chuẩn hóa tên cột.
campaign = pd.read_csv(MARKETING_CSV, sep=";")
campaign.columns = campaign.columns.str.strip()

# Bước 2: xem kích thước, tên cột và một số dòng đầu để hiểu cấu trúc bảng.
print(f"Số dòng: {campaign.shape[0]:,}; số cột: {campaign.shape[1]:,}")
print("Tên cột:", campaign.columns.tolist())
display(campaign.head())

# Bước 3: tổng hợp kiểu dữ liệu, giá trị thiếu, tỷ lệ thiếu và số giá trị phân biệt.
quality_report = pd.DataFrame(
    {
        "dtype": campaign.dtypes.astype(str),
        "missing_count": campaign.isna().sum(),
        "missing_percent": (campaign.isna().mean() * 100).round(2),
        "distinct_count": campaign.nunique(dropna=True),
    }
)
display(quality_report.sort_values("missing_percent", ascending=False))
print(f"Số dòng trùng hoàn toàn: {campaign.duplicated().sum():,}")

# Bước 4: kiểm tra nhãn đích và xem phân bố phản hồi của chiến dịch.
if "Response" not in campaign.columns:
    raise ValueError(
        f"Không tìm thấy cột Response. Các cột hiện có: {campaign.columns.tolist()}"
    )
response_counts = campaign["Response"].value_counts(dropna=False).sort_index()
response_summary = pd.DataFrame(
    {
        "count": response_counts,
        "percent": (response_counts / len(campaign) * 100).round(2),
    }
)
print("Phân bố phản hồi chiến dịch (Response):")
display(response_summary)

# Tóm tắt hiệu quả phản hồi và đặc trưng khách hàng theo nhóm Response.
campaign_summary = campaign.groupby("Response").agg(
    customers=("ID", "count"),
    mean_income=("Income", "mean"),
    mean_recency=("Recency", "mean"),
    mean_wine_spend=("MntWines", "mean"),
    mean_web_purchases=("NumWebPurchases", "mean"),
).round(2)
display(campaign_summary)
print(f"Tỷ lệ phản hồi chung: {campaign['Response'].mean() * 100:.2f}%")


Số dòng: 2,240; số cột: 29
Tên cột: ['ID', 'Year_Birth', 'Education', 'Marital_Status', 'Income', 'Kidhome', 'Teenhome', 'Dt_Customer', 'Recency', 'MntWines', 'MntFruits', 'MntMeatProducts', 'MntFishProducts', 'MntSweetProducts', 'MntGoldProds', 'NumDealsPurchases', 'NumWebPurchases', 'NumCatalogPurchases', 'NumStorePurchases', 'NumWebVisitsMonth', 'AcceptedCmp3', 'AcceptedCmp4', 'AcceptedCmp5', 'AcceptedCmp1', 'AcceptedCmp2', 'Complain', 'Z_CostContact', 'Z_Revenue', 'Response']


,ID,Year_Birth,Education,Marital_Status,Income,Kidhome,Teenhome,Dt_Customer,Recency,MntWines,...,NumWebVisitsMonth,AcceptedCmp3,AcceptedCmp4,AcceptedCmp5,AcceptedCmp1,AcceptedCmp2,Complain,Z_CostContact,Z_Revenue,Response
0,5524,1957,Graduation,Single,58138.0,0,0,2012-09-04,58,635,...,7,0,0,0,0,0,0,3,11,1
1,2174,1954,Graduation,Single,46344.0,1,1,2014-03-08,38,11,...,5,0,0,0,0,0,0,3,11,0
2,4141,1965,Graduation,Together,71613.0,0,0,2013-08-21,26,426,...,4,0,0,0,0,0,0,3,11,0
3,6182,1984,Graduation,Together,26646.0,1,0,2014-02-10,26,11,...,6,0,0,0,0,0,0,3,11,0
4,5324,1981,PhD,Married,58293.0,1,0,2014-01-19,94,173,...,5,0,0,0,0,0,0,3,11,0


,dtype,missing_count,missing_percent,distinct_count
Income,float64,24,1.07,1974
ID,int64,0,0.00,2240
Year_Birth,int64,0,0.00,59
Education,str,0,0.00,5
Marital_Status,str,0,0.00,8
Kidhome,int64,0,0.00,3
Teenhome,int64,0,0.00,3
Dt_Customer,str,0,0.00,663
Recency,int64,0,0.00,100
MntWines,int64,0,0.00,776


Số dòng trùng hoàn toàn: 0
Phân bố phản hồi chiến dịch (Response):


,count,percent
Response,,
0,1906,85.09
1,334,14.91


,customers,mean_income,mean_recency,mean_wine_spend,mean_web_purchases
Response,,,,,
0,1906,50839.13,51.51,269.1,3.91
1,334,60209.68,35.38,502.7,5.07


Tỷ lệ phản hồi chung: 14.91%


In [6]:
# Bước 1: kiểm tra target và nạp SweetViz; thiếu thư viện sẽ báo lỗi import cụ thể.
if "Response" not in campaign.columns:
    raise ValueError(
        f"Không tìm thấy cột Response. Các cột hiện có: {campaign.columns.tolist()}"
    )
import sweetviz as sv

# Bước 2: tạo báo cáo, dùng Response làm biến mục tiêu chiến dịch.
report = sv.analyze(campaign, target_feat="Response")

# Bước 3: lưu báo cáo trong thư mục reports cạnh notebook; không tự mở trình duyệt.
report_path = BASE_DIR / "reports" / "marketing_campaign_sweetviz.html"
report_path.parent.mkdir(parents=True, exist_ok=True)
report.show_html(filepath=str(report_path), open_browser=False)
print(f"Đã tạo báo cáo SweetViz: {report_path.resolve()}")


Done! Use 'show' commands to display/save.   |██████████| [100%]   00:01 -> (00:00 left)

Report D:\ML\sgu26k2_ML\Lab05\reports\marketing_campaign_sweetviz.html was generated.
Đã tạo báo cáo SweetViz: D:\ML\sgu26k2_ML\Lab05\reports\marketing_campaign_sweetviz.html
